In [2]:
import re, zipfile, glob, tempfile
from pathlib import Path
from collections import Counter
import cv2

ZIP = next(Path(p) for p in glob.glob(r"D:\Downloads\*.zip") if "UAlpha40" in Path(p).name)
zf = zipfile.ZipFile(ZIP)
names = [n for n in zf.namelist() if not n.endswith("/")]

DYN = ["2-Hay", "Alifmad", "Aray", "Jeem"]
vid = {}
for d in DYN:
    files = [n for n in names if f"/{d}/" in n]
    vid[d] = files
    mb = sum(zf.getinfo(n).file_size for n in files) / 1e6
    has_id = sum(bool(re.match(r"s\d{4}", Path(n).name, re.I)) for n in files)
    print(f"{d:8s} {len(files):3d} files | {mb:6.0f} MB | {dict(Counter(Path(n).suffix.lower() for n in files))}"
          f" | sXXXX wale: {has_id}")
    print("     namune:", [Path(n).name for n in files[:3]])

# --- har harf ki ek video kholkar dekho ---
print()
for d in DYN:
    n0 = vid[d][0]
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp) / Path(n0).name
        p.write_bytes(zf.read(n0))
        cap = cv2.VideoCapture(str(p))
        print(f"{d:8s} khula: {cap.isOpened()} | frames: {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}"
              f" | fps: {cap.get(cv2.CAP_PROP_FPS):.0f}"
              f" | size: {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
        cap.release()

2-Hay     87 files |    151 MB | {'.mp4': 87} | sXXXX wale: 86
     namune: ['s0253.mp4', 's0261.mp4', 's0257.mp4']
Alifmad  105 files |    137 MB | {'.mp4': 105} | sXXXX wale: 105
     namune: ['s0261.mp4', 's0254.mp4', 's0262.mp4']
Aray     102 files |    140 MB | {'.mp4': 102} | sXXXX wale: 102
     namune: ['s0255.mp4', 's0252.mp4', 's0260.mp4']
Jeem      99 files |    142 MB | {'.mp4': 99} | sXXXX wale: 99
     namune: ['s0257.mp4', 's0255.mp4', 's0252.mp4']

2-Hay    khula: True | frames: 64 | fps: 30 | size: 1920x1080
Alifmad  khula: True | frames: 43 | fps: 30 | size: 1920x1080
Aray     khula: True | frames: 61 | fps: 30 | size: 1920x1080
Jeem     khula: True | frames: 61 | fps: 30 | size: 1920x1080


In [3]:
import numpy as np, pandas as pd, time, sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.landmarks import extract_landmarks

def sid(x):
    return re.search(r"(\d+)", str(x)).group(1).zfill(4)

train = pd.read_csv(ROOT / "data/processed/train.csv")
test = pd.read_csv(ROOT / "data/processed/test.csv")
train_sig = {sid(s) for s in train["signer_id"]}
test_sig = {sid(s) for s in test["signer_id"]}

rows = []
for d in DYN:
    for n in vid[d]:
        m = re.match(r"s(\d{4})", Path(n).name, re.I)
        rows.append({"zip_path": n, "label": d, "name": Path(n).name,
                     "signer_id": m.group(1) if m else None})
vdf = pd.DataFrame(rows)
vdf["split"] = np.where(vdf.signer_id.isin(train_sig), "train",
                np.where(vdf.signer_id.isin(test_sig), "test", "none"))

print("sXXXX nahi wali video:", vdf[vdf.signer_id.isna()].name.tolist())
print(vdf.groupby(["label", "split"]).size().unstack(fill_value=0))
print("ek signer ki ek harf ki max videos:", vdf.groupby(["label", "signer_id"]).size().max())

# --- ek video ke landmarks aur waqt ---
def video_landmarks(zip_path, step=2, width=640):
    out = []
    with tempfile.TemporaryDirectory() as tmp:
        vp, fp = Path(tmp) / "v.mp4", Path(tmp) / "f.jpg"
        vp.write_bytes(zf.read(zip_path))
        cap = cv2.VideoCapture(str(vp))
        i = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if i % step == 0:
                h, w = frame.shape[:2]
                frame = cv2.resize(frame, (width, int(h * width / w)))
                cv2.imwrite(str(fp), frame)
                out.append((i, extract_landmarks(fp)))
            i += 1
        cap.release()
    return out

t0 = time.time()
res = video_landmarks(vid["Jeem"][0])
print(f"\n{time.time() - t0:.1f} sec | frames process hue: {len(res)} | haath mila: {sum(lm is not None for _, lm in res)}")

sXXXX nahi wali video: ['s316.mp4']
split    none  test  train
label                     
2-Hay      17    14     56
Alifmad    26    15     64
Aray       22    16     64
Jeem       21    16     62
ek signer ki ek harf ki max videos: 2

1.7 sec | frames process hue: 31 | haath mila: 31


In [4]:
# s316 jaise 3-digit naam bhi pakadne ke liye
vdf["signer_id"] = vdf["name"].str.extract(r"^s(\d+)", flags=re.I)[0].str.zfill(4)
vdf["split"] = np.where(vdf.signer_id.isin(test_sig), "test", "train")     # none -> train
print(vdf.groupby(["label", "split"]).size().unstack(fill_value=0))
print("signer_id khali:", vdf.signer_id.isna().sum())

parts_dir = ROOT / "data/processed/video_parts"
parts_dir.mkdir(parents=True, exist_ok=True)
cols = ["label", "signer_id", "name", "split", "frame_idx"] + [f"f{i}" for i in range(63)]

t0 = time.time()
for label, grp in vdf.groupby("label"):
    f = parts_dir / f"{label}.csv"
    if f.exists():
        continue                                           # pehle ho chuka, chhor do (resume)
    rows = []
    for r in grp.itertuples():
        for i, lm in video_landmarks(r.zip_path):
            vals = list(lm) if lm is not None else [np.nan] * 63
            rows.append([r.label, r.signer_id, r.name, r.split, i] + vals)
    pd.DataFrame(rows, columns=cols).to_csv(f, index=False)
    print(f"{label:8s} {len(grp)} videos | {time.time() - t0:.0f} sec")

vl = pd.concat([pd.read_csv(f, dtype={"signer_id": str}) for f in sorted(parts_dir.glob("*.csv"))],
               ignore_index=True)
vl.to_csv(ROOT / "data/processed/video_landmarks.csv", index=False)
lm_cols = [f"f{i}" for i in range(63)]
print("\nkul rows:", vl.shape, "| videos:", vl.groupby(["label", "name"]).ngroups)
print("haath mila (frames ka hissa):", round(vl[lm_cols].notna().all(axis=1).mean(), 3))

split    test  train
label               
2-Hay      14     73
Alifmad    15     90
Aray       16     86
Jeem       16     83
signer_id khali: 0
2-Hay    87 videos | 149 sec
Alifmad  105 videos | 288 sec
Aray     102 videos | 436 sec
Jeem     99 videos | 580 sec

kul rows: (11439, 68) | videos: 393
haath mila (frames ka hissa): 0.965


In [5]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from src.features import normalize_landmarks

lm_cols = [f"f{i}" for i in range(63)]

def video_sequence(g):
    """Ek video ke frames -> (n, 65): 63 normalized shape + 2 wrist harkat."""
    g = g.sort_values("frame_idx")
    arr = g[lm_cols].interpolate(limit_direction="both").to_numpy(dtype=np.float64)
    if np.isnan(arr).any():
        return None                                   # kisi frame mein bhi haath nahi mila
    P = arr.reshape(len(arr), 21, 3)
    size = np.linalg.norm(P[:, 9] - P[:, 0], axis=1).mean() + 1e-6
    wrist = (P[:, 0, :2] - P[0, 0, :2]) / size        # pehle frame se kalai ki harkat
    shape = np.vstack([normalize_landmarks(r) for r in arr])
    return np.hstack([shape, wrist])

def resample(seq, T):
    t_old = np.linspace(0, 1, len(seq))
    t_new = np.linspace(0, 1, T)
    return np.column_stack([np.interp(t_new, t_old, seq[:, j]) for j in range(seq.shape[1])])

seqs, meta = [], []
for (label, name), g in vl.groupby(["label", "name"]):
    s = video_sequence(g)
    if s is None:
        continue
    seqs.append(s)
    meta.append({"label": label, "name": name,
                 "signer_id": g.signer_id.iloc[0], "split": g.split.iloc[0]})
meta = pd.DataFrame(meta)
print("kamyab videos:", len(meta), "/ 393 | har video ke frames:",
      min(len(s) for s in seqs), "se", max(len(s) for s in seqs))

is_train = (meta.split == "train").to_numpy()
y = meta.label.astype(str).to_numpy()
groups = meta.signer_id.astype(str).to_numpy()

models = {
    "Baseline":  DummyClassifier(strategy="most_frequent"),
    "KNN (k=3)": Pipeline([("sc", StandardScaler()), ("m", KNeighborsClassifier(3))]),
    "SVM RBF":   Pipeline([("sc", StandardScaler()), ("m", SVC(kernel="rbf", C=10, gamma="scale"))]),
    "RF":        RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "MLP":       Pipeline([("sc", StandardScaler()),
                           ("m", MLPClassifier((100, 50), max_iter=500, random_state=42))]),
}
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for T in (8, 16):
    X = np.vstack([resample(s, T).ravel() for s in seqs])
    for name, m in models.items():
        r = cross_validate(m, X[is_train], y[is_train], groups=groups[is_train], cv=cv,
                           scoring={"acc": "accuracy", "f1": "f1_macro"})
        rows.append({"T": T, "model": name, "n_features": X.shape[1],
                     "acc": r["test_acc"].mean(), "macro_f1": r["test_f1"].mean()})
res6 = pd.DataFrame(rows)
print(res6.round(4).to_string(index=False))
res6.to_csv(ROOT / "results/advanced_video_cv.csv", index=False)

kamyab videos: 393 / 393 | har video ke frames: 7 se 59
 T     model  n_features    acc  macro_f1
 8  Baseline         520 0.2710    0.1066
 8 KNN (k=3)         520 0.9670    0.9666
 8   SVM RBF         520 0.9760    0.9756
 8        RF         520 0.9910    0.9906
 8       MLP         520 0.9850    0.9846
16  Baseline        1040 0.2710    0.1066
16 KNN (k=3)        1040 0.9580    0.9576
16   SVM RBF        1040 0.9699    0.9692
16        RF        1040 0.9879    0.9874
16       MLP        1040 0.9792    0.9787


In [6]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import joblib

is_test = (meta.split == "test").to_numpy()
rows, fitted = [], {}
for T in (8, 16):
    X = np.vstack([resample(s, T).ravel() for s in seqs])
    for name, m0 in models.items():
        m = clone(m0).fit(X[is_train], y[is_train])
        p = m.predict(X[is_test])
        rows.append({"T": T, "model": name,
                     "test_acc": accuracy_score(y[is_test], p),
                     "test_macro_f1": f1_score(y[is_test], p, average="macro")})
        fitted[(T, name)] = (m, p)

res7 = pd.DataFrame(rows)
print(res7.round(4).to_string(index=False))
res7.to_csv(ROOT / "results/advanced_video_test.csv", index=False)

# CV se chuna hua: RF, T=8
m_best, p_best = fitted[(8, "RF")]
labels = sorted(set(y))
print("\nConfusion matrix (RF, T=8, test videos):")
print(pd.DataFrame(confusion_matrix(y[is_test], p_best, labels=labels),
                   index=labels, columns=labels))
print(classification_report(y[is_test], p_best, digits=3))
joblib.dump(m_best, ROOT / "models/dynamic_rf_T8.joblib")

 T     model  test_acc  test_macro_f1
 8  Baseline    0.2459         0.0987
 8 KNN (k=3)    0.9672         0.9670
 8   SVM RBF    0.9836         0.9832
 8        RF    0.9672         0.9661
 8       MLP    0.9836         0.9832
16  Baseline    0.2459         0.0987
16 KNN (k=3)    0.9508         0.9513
16   SVM RBF    0.9836         0.9832
16        RF    0.9672         0.9661
16       MLP    0.9836         0.9832

Confusion matrix (RF, T=8, test videos):
         2-Hay  Alifmad  Aray  Jeem
2-Hay       12        0     2     0
Alifmad      0       15     0     0
Aray         0        0    16     0
Jeem         0        0     0    16
              precision    recall  f1-score   support

       2-Hay      1.000     0.857     0.923        14
     Alifmad      1.000     1.000     1.000        15
        Aray      0.889     1.000     0.941        16
        Jeem      1.000     1.000     1.000        16

    accuracy                          0.967        61
   macro avg      0.972     0.964 

['d:\\CS 5 Course\\ML(Lab)\\Project\\psl-translator\\models\\dynamic_rf_T8.joblib']